In [1]:
import pandas as pd

In [4]:
#df = pd.read_json(r"C:\dev\google-timeline\location-history (1).json").drop("timelineMemory", axis=1)

df = pd.read_json(r"C:\dev\google-and-spotify\pipeline\raw\google\location-history.json").drop("timelineMemory", axis=1)

df["startTime"] = pd.to_datetime(df["startTime"], utc=True)
df["endTime"] = pd.to_datetime(df["endTime"], utc=True)

df["startTime"] = df["startTime"].dt.tz_convert("America/Sao_Paulo")
df["endTime"] = df["endTime"].dt.tz_convert("America/Sao_Paulo")

#df = df[df["startTime"] >= "2024-12-01"]

df.shape

FileNotFoundError: File C:\dev\google-and-spotify\pipeline\raw\google\location-history.json does not exist

In [ ]:
df.head()

,endTime,startTime,visit,activity,timelinePath
0,2014-04-28 15:49:24.902000-03:00,2014-04-28 14:02:23.741000-03:00,"{'hierarchyLevel': '0', 'topCandidate': {'prob...",NaN,NaN
1,2014-05-11 12:11:08.165000-03:00,2014-05-11 11:38:21.854000-03:00,"{'hierarchyLevel': '0', 'topCandidate': {'prob...",NaN,NaN
2,2014-05-13 17:08:40.429000-03:00,2014-05-12 14:28:38.974000-03:00,"{'hierarchyLevel': '0', 'topCandidate': {'prob...",NaN,NaN
3,2014-05-16 18:57:03.590000-03:00,2014-05-16 15:17:42.576000-03:00,"{'hierarchyLevel': '0', 'topCandidate': {'prob...",NaN,NaN
4,2014-05-17 05:53:11.258000-03:00,2014-05-17 05:33:00.273000-03:00,"{'hierarchyLevel': '0', 'topCandidate': {'prob...",NaN,NaN


In [ ]:
pd.set_option("display.max_rows", None)
df.groupby(by=[df["endTime"].dt.year, df["endTime"].dt.month]).size()

endTime  endTime
2014     4            2
         5           12
         6           25
         7           15
         8            5
         9           45
         10         122
         11         100
         12          93
2015     1           52
         2           49
         3          248
         4          281
         5          363
         6          339
         7          241
         8           51
         9           73
         10         149
         11          54
         12         155
2016     1          212
         2          302
         3          167
         4          114
         5           82
         6            1
         7           51
         8          534
         9          653
         10         608
         11         500
         12         361
2017     1          408
         2          441
         3          545
         4          574
         5          503
         6          435
         7          474
         8          489

In [ ]:
df.isna().sum()

endTime             0
startTime           0
visit           30293
activity        31570
timelinePath    26661
dtype: int64

In [ ]:
mask = (df["timelinePath"].isna()) & (df["visit"].isna()) & (df["activity"].isna())
df[mask].shape

(106, 5)

In [ ]:
mask = ~(df["timelinePath"].isna()) & ~(df["visit"].isna()) & ~(df["activity"].isna())
df[mask].shape

(0, 5)

In [ ]:
mask = ~(df["visit"].isna()) & (df["activity"].isna()) & (df["timelinePath"].isna())
df_visit = df[mask][["startTime", "endTime", "visit"]]

mask = (df["visit"].isna()) & ~(df["activity"].isna()) & (df["timelinePath"].isna())
df_activity = df[mask][["startTime", "endTime", "activity"]]

mask = (df["visit"].isna()) & (df["activity"].isna()) & ~(df["timelinePath"].isna())
df_timeline = df[mask][["startTime", "endTime", "timelinePath"]]

In [ ]:
df_visit.iloc[1]["visit"]

{'hierarchyLevel': '0',
 'topCandidate': {'probability': '0.291292',
  'semanticType': 'Work',
  'placeID': 'ChIJ2e77KABpGZURe_fLtv-W7Tc',
  'placeLocation': 'geo:-29.776378,-51.137439'},
 'probability': '0.750000'}

In [ ]:
df_activity.iloc[12600]["activity"]

{'probability': '0.987389',
 'end': 'geo:-29.769856,-51.146069',
 'topCandidate': {'type': 'in passenger vehicle', 'probability': '0.926526'},
 'distanceMeters': '13653.925781',
 'start': 'geo:-29.875989,-51.177754'}

In [ ]:
df_timeline.iloc[1]["timelinePath"]

[{'point': 'geo:-29.769324,-51.137160',
  'durationMinutesOffsetFromStartTime': '38'}]

In [ ]:
visit_df = pd.json_normalize(df_visit["visit"])

df_visit = pd.concat(
    [df_visit.drop(columns=["visit"]), visit_df],
    axis=1
)

df_visit[["topCandidate.latitude", "topCandidate.longitude"]] = (
    df_visit["topCandidate.placeLocation"]
    .str.replace("geo:", "")
    .str.split(",", expand=True)
)

df_visit["topCandidate.latitude"] = df_visit["topCandidate.latitude"].astype(float)
df_visit["topCandidate.longitude"] = df_visit["topCandidate.longitude"].astype(float)

df_visit = df_visit.drop("topCandidate.placeLocation", axis=1)

In [ ]:
activity_df = pd.json_normalize(df_activity["activity"])
activity_df.head()

,end,distanceMeters,start,topCandidate.type,topCandidate.probability,probability
0,"geo:-29.768794,-51.144968",281.000000,"geo:-29.768794,-51.144968",unknown,0.000000,NaN
1,"geo:-29.766655,-51.143866",260.474091,"geo:-29.768794,-51.144968",unknown,0.000000,NaN
2,"geo:-29.797370,-51.151313",89.000000,"geo:-29.797178,-51.151324",walking,0.000000,NaN
3,"geo:-29.768900,-51.145004",3149.000000,"geo:-29.797370,-51.151313",unknown,0.000000,NaN
4,"geo:-29.761488,-51.145589",785.000000,"geo:-29.768794,-51.144968",unknown,0.000000,NaN


In [ ]:
activity_df.shape

(12639, 6)

In [ ]:
df_activity.shape

(12639, 3)

In [ ]:
df_activity = pd.concat(
    [
        df_activity.drop(columns=["activity"]).reset_index(drop=True),
        activity_df.reset_index(drop=True)
    ],
    axis=1
)

In [ ]:
df_activity.shape

(12639, 8)

In [ ]:

# Separar coordenadas do início
df_activity[["start_latitude", "start_longitude"]] = (
    df_activity["start"]
    .str.replace("geo:", "")
    .str.split(",", expand=True)
)

# Separar coordenadas do fim
df_activity[["end_latitude", "end_longitude"]] = (
    df_activity["end"]
    .str.replace("geo:", "")
    .str.split(",", expand=True)
)

# Converter para float
cols = [
    "start_latitude",
    "start_longitude",
    "end_latitude",
    "end_longitude"
]

df_activity[cols] = df_activity[cols].astype(float)

# Converter distância para número
df_activity["distanceMeters"] = (
    df_activity["distanceMeters"]
    .astype(float)
)

# Renomear colunas do topCandidate
df_activity = df_activity.rename(columns={
    "topCandidate.type": "activity_type",
    "topCandidate.probability": "activity_probability"
})

# Remover colunas originais
df_activity = df_activity.drop(
    columns=["start", "end"]
)


In [ ]:
# Guardar o índice original
timeline_df = df_timeline.copy()
timeline_df["activity_id"] = timeline_df.index

# Explodir a lista de pontos
timeline_df = timeline_df.explode("timelinePath")

timeline_df = timeline_df.reset_index(drop=True)

timeline_df = pd.concat(
    [
        timeline_df.drop(columns=["timelinePath"]),
        pd.json_normalize(timeline_df["timelinePath"])
    ],
    axis=1
)

# Separar latitude e longitude
timeline_df[["latitude", "longitude"]] = (
    timeline_df["point"]
    .str.replace("geo:", "")
    .str.split(",", expand=True)
)

# Converter tipos
timeline_df["latitude"] = timeline_df["latitude"].astype(float)
timeline_df["longitude"] = timeline_df["longitude"].astype(float)

timeline_df["duration_minutes_offset"] = (
    timeline_df["durationMinutesOffsetFromStartTime"]
    .astype(int)
)

# Remover colunas desnecessárias
timeline_df = timeline_df.drop(
    columns=[
        "point",
        "durationMinutesOffsetFromStartTime"
    ]
)

timeline_df = timeline_df.reset_index(drop=True)

df_timeline = timeline_df

In [ ]:
df_activity.shape

(12639, 10)

In [ ]:
df_visit.head()

,startTime,endTime,hierarchyLevel,probability,topCandidate.probability,topCandidate.semanticType,topCandidate.placeID,topCandidate.latitude,topCandidate.longitude
0,2014-04-28 14:02:23.741000-03:00,2014-04-28 15:49:24.902000-03:00,0,0.770000,0.313333,Unknown,ChIJ14C6tlFoGZURwuDID3ysPqY,-29.768567,-51.145016
1,2014-05-11 11:38:21.854000-03:00,2014-05-11 12:11:08.165000-03:00,0,0.750000,0.291292,Work,ChIJ2e77KABpGZURe_fLtv-W7Tc,-29.776378,-51.137439
2,2014-05-12 14:28:38.974000-03:00,2014-05-13 17:08:40.429000-03:00,0,0.640000,0.223375,Unknown,ChIJ14C6tlFoGZURwuDID3ysPqY,-29.768567,-51.145016
3,2014-05-16 15:17:42.576000-03:00,2014-05-16 18:57:03.590000-03:00,0,0.770000,0.313505,Unknown,ChIJ14C6tlFoGZURwuDID3ysPqY,-29.768567,-51.145016
4,2014-05-17 05:33:00.273000-03:00,2014-05-17 05:53:11.258000-03:00,0,0.750000,0.066889,Work,ChIJ2e77KABpGZURe_fLtv-W7Tc,-29.776378,-51.137439


In [ ]:
df_activity.shape

(12639, 10)

In [ ]:
df_activity.head()

,startTime,endTime,distanceMeters,activity_type,activity_probability,probability,start_latitude,start_longitude,end_latitude,end_longitude
0,2014-09-07 08:30:34.088000-03:00,2014-09-07 09:17:46.566000-03:00,281.000000,unknown,0.000000,NaN,-29.768794,-51.144968,-29.768794,-51.144968
1,2014-09-07 13:51:33.819000-03:00,2014-09-07 14:07:48.824000-03:00,260.474091,unknown,0.000000,NaN,-29.768794,-51.144968,-29.766655,-51.143866
2,2014-09-07 14:54:09.592000-03:00,2014-09-07 15:00:44.824000-03:00,89.000000,walking,0.000000,NaN,-29.797178,-51.151324,-29.797370,-51.151313
3,2014-09-07 15:00:44.824000-03:00,2014-09-07 15:24:00.732000-03:00,3149.000000,unknown,0.000000,NaN,-29.797370,-51.151313,-29.768900,-51.145004
4,2014-10-03 13:48:19.886000-03:00,2014-10-03 13:51:32.171000-03:00,785.000000,unknown,0.000000,NaN,-29.768794,-51.144968,-29.761488,-51.145589


In [ ]:
df_timeline.tail(60)

,startTime,endTime,activity_id,latitude,longitude,duration_minutes_offset
160587,2026-08-01 21:00:00-03:00,2026-08-01 23:00:00-03:00,44200,-30.012695,-51.201379,12
160588,2026-08-01 21:00:00-03:00,2026-08-01 23:00:00-03:00,44200,-30.010518,-51.202391,111
160589,2026-08-01 21:00:00-03:00,2026-08-01 23:00:00-03:00,44200,-30.012463,-51.195811,113
160590,2026-08-01 21:00:00-03:00,2026-08-01 23:00:00-03:00,44200,-30.012704,-51.187888,115
160591,2026-08-01 23:00:00-03:00,2026-08-02 01:00:00-03:00,44201,-30.039434,-51.176785,2
160592,2026-08-01 23:00:00-03:00,2026-08-02 01:00:00-03:00,44201,-30.038903,-51.173072,4
160593,2026-08-01 23:00:00-03:00,2026-08-02 01:00:00-03:00,44201,-30.052394,-51.171017,6
160594,2026-08-01 23:00:00-03:00,2026-08-02 01:00:00-03:00,44201,-30.052576,-51.170907,9
160595,2026-08-01 23:00:00-03:00,2026-08-02 01:00:00-03:00,44201,-30.051885,-51.169574,10
160596,2026-08-01 23:00:00-03:00,2026-08-02 01:00:00-03:00,44201,-30.051579,-51.169132,11


In [ ]:
from sqlalchemy import create_engine

DATABASE_URL = (
    "postgresql://postgres:1234"
    "@localhost:5432/google_and_spotify"
)


engine = create_engine(
    DATABASE_URL
)

with engine.connect() as conn:
    print("Conectado!")

Conectado!


In [ ]:
# ==========================
# Renomear colunas originais
# ==========================
df = df_visit.copy()
df = df.rename(
    columns={
        "startTime": "start_time",
        "endTime": "end_time",
        "topCandidate.placeID": "candidate_id",
        "topCandidate.semanticType": "semantic_type",
        "topCandidate.latitude": "latitude",
        "topCandidate.longitude": "longitude",
        "topCandidate.probability": "candidate_probability",
        "probability": "probability"
    }
)


# ==========================
# Criar dimensão de candidatos
# ==========================

df_candidates = (
    df[
        [
            "candidate_id",
            "semantic_type",
            "latitude",
            "longitude"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


# ==========================
# Criar fato de visitas
# ==========================

df_visit = df[
    [
        "start_time",
        "end_time",
        "candidate_id",
        "probability",
        "candidate_probability"
    ]
].copy()


# duração da visita em minutos

df_visit["duration_minutes"] = (
    (
        df_visit["end_time"]
        -
        df_visit["start_time"]
    )
    .dt.total_seconds()
    / 60
).astype(float)


# Reordenar colunas conforme tabela

df_visit = df_visit[
    [
        "start_time",
        "end_time",
        "duration_minutes",
        "candidate_id",
        "probability",
        "candidate_probability"
    ]
]

In [ ]:
df_visit.head(5)

,start_time,end_time,duration_minutes,candidate_id,probability,candidate_probability
0,2014-04-28 14:02:23.741000-03:00,2014-04-28 15:49:24.902000-03:00,107.019350,ChIJ14C6tlFoGZURwuDID3ysPqY,0.770000,0.313333
1,2014-05-11 11:38:21.854000-03:00,2014-05-11 12:11:08.165000-03:00,32.771850,ChIJ2e77KABpGZURe_fLtv-W7Tc,0.750000,0.291292
2,2014-05-12 14:28:38.974000-03:00,2014-05-13 17:08:40.429000-03:00,1600.024250,ChIJ14C6tlFoGZURwuDID3ysPqY,0.640000,0.223375
3,2014-05-16 15:17:42.576000-03:00,2014-05-16 18:57:03.590000-03:00,219.350233,ChIJ14C6tlFoGZURwuDID3ysPqY,0.770000,0.313505
4,2014-05-17 05:33:00.273000-03:00,2014-05-17 05:53:11.258000-03:00,20.183083,ChIJ2e77KABpGZURe_fLtv-W7Tc,0.750000,0.066889


In [ ]:
df_candidates.head()

,candidate_id,semantic_type,latitude,longitude
0,ChIJ14C6tlFoGZURwuDID3ysPqY,Unknown,-29.768567,-51.145016
1,ChIJ2e77KABpGZURe_fLtv-W7Tc,Work,-29.776378,-51.137439
2,ChIJu3JfIE5oGZUR0DsdD36N4iQ,Unknown,-29.766867,-51.143465
3,ChIJyW9ltlpoGZURswCVpj8lN8I,Inferred Home,-29.768794,-51.144968
4,ChIJ4alkZZ94GJURWLEZAix0Q8k,Unknown,-29.759402,-50.020339


In [ ]:
df_candidates = (
    df_candidates
    .drop_duplicates(subset=["candidate_id"])
)

df_candidates.to_sql("dim_candidates", con=engine, if_exists="append", index=False)

869

In [ ]:
df_visit.head()

,start_time,end_time,duration_minutes,candidate_id,probability,candidate_probability
0,2014-04-28 14:02:23.741000-03:00,2014-04-28 15:49:24.902000-03:00,107.019350,ChIJ14C6tlFoGZURwuDID3ysPqY,0.770000,0.313333
1,2014-05-11 11:38:21.854000-03:00,2014-05-11 12:11:08.165000-03:00,32.771850,ChIJ2e77KABpGZURe_fLtv-W7Tc,0.750000,0.291292
2,2014-05-12 14:28:38.974000-03:00,2014-05-13 17:08:40.429000-03:00,1600.024250,ChIJ14C6tlFoGZURwuDID3ysPqY,0.640000,0.223375
3,2014-05-16 15:17:42.576000-03:00,2014-05-16 18:57:03.590000-03:00,219.350233,ChIJ14C6tlFoGZURwuDID3ysPqY,0.770000,0.313505
4,2014-05-17 05:33:00.273000-03:00,2014-05-17 05:53:11.258000-03:00,20.183083,ChIJ2e77KABpGZURe_fLtv-W7Tc,0.750000,0.066889


In [ ]:
df_visit.to_sql("fact_visit", con=engine, if_exists="append", index=False)

822

In [ ]:
df_activity.isna().sum()

startTime                   0
endTime                     0
distanceMeters              0
activity_type               0
activity_probability        0
probability             11102
start_latitude              0
start_longitude             0
end_latitude                0
end_longitude               0
dtype: int64

In [ ]:
df_activity.head()

,startTime,endTime,distanceMeters,activity_type,activity_probability,probability,start_latitude,start_longitude,end_latitude,end_longitude
0,2014-09-07 08:30:34.088000-03:00,2014-09-07 09:17:46.566000-03:00,281.000000,unknown,0.000000,NaN,-29.768794,-51.144968,-29.768794,-51.144968
1,2014-09-07 13:51:33.819000-03:00,2014-09-07 14:07:48.824000-03:00,260.474091,unknown,0.000000,NaN,-29.768794,-51.144968,-29.766655,-51.143866
2,2014-09-07 14:54:09.592000-03:00,2014-09-07 15:00:44.824000-03:00,89.000000,walking,0.000000,NaN,-29.797178,-51.151324,-29.797370,-51.151313
3,2014-09-07 15:00:44.824000-03:00,2014-09-07 15:24:00.732000-03:00,3149.000000,unknown,0.000000,NaN,-29.797370,-51.151313,-29.768900,-51.145004
4,2014-10-03 13:48:19.886000-03:00,2014-10-03 13:51:32.171000-03:00,785.000000,unknown,0.000000,NaN,-29.768794,-51.144968,-29.761488,-51.145589


In [ ]:
df_activity = df_activity.rename(
    columns={
        "startTime": "start_time",
        "endTime": "end_time",
        "distanceMeters": "distance_meters",
    }
)
df_activity = df_activity.drop("probability", axis=1)

In [ ]:
df_activity.head()

,start_time,end_time,distance_meters,activity_type,activity_probability,start_latitude,start_longitude,end_latitude,end_longitude
0,2014-09-07 08:30:34.088000-03:00,2014-09-07 09:17:46.566000-03:00,281.000000,unknown,0.000000,-29.768794,-51.144968,-29.768794,-51.144968
1,2014-09-07 13:51:33.819000-03:00,2014-09-07 14:07:48.824000-03:00,260.474091,unknown,0.000000,-29.768794,-51.144968,-29.766655,-51.143866
2,2014-09-07 14:54:09.592000-03:00,2014-09-07 15:00:44.824000-03:00,89.000000,walking,0.000000,-29.797178,-51.151324,-29.797370,-51.151313
3,2014-09-07 15:00:44.824000-03:00,2014-09-07 15:24:00.732000-03:00,3149.000000,unknown,0.000000,-29.797370,-51.151313,-29.768900,-51.145004
4,2014-10-03 13:48:19.886000-03:00,2014-10-03 13:51:32.171000-03:00,785.000000,unknown,0.000000,-29.768794,-51.144968,-29.761488,-51.145589


In [ ]:
df_activity.shape

(12639, 9)

In [ ]:
df_activity.to_sql("fact_activity", con=engine, if_exists="append", index=False)

639

In [ ]:
df_timeline.head()

,startTime,endTime,activity_id,latitude,longitude,duration_minutes_offset
0,2014-04-28 13:00:00-03:00,2014-04-28 15:00:00-03:00,26555,-29.768703,-51.144916,62
1,2014-05-11 11:00:00-03:00,2014-05-11 13:00:00-03:00,26556,-29.769324,-51.137160,38
2,2014-05-12 13:00:00-03:00,2014-05-12 15:00:00-03:00,26557,-29.768741,-51.144985,89
3,2014-05-16 15:00:00-03:00,2014-05-16 17:00:00-03:00,26558,-29.768676,-51.144905,18
4,2014-05-17 05:00:00-03:00,2014-05-17 07:00:00-03:00,26559,-29.768442,-51.144716,33


In [ ]:
df_timeline = df_timeline.rename(
    columns={
        "startTime": "start_time",
        "endTime": "end_time"
    }
)
df_timeline.drop("activity_id", axis=1, inplace=True)

In [ ]:
df_timeline.to_sql("timeline_path", con=engine, if_exists="append", index=False)

647